# TripMe AI Mode - Fine-tune a Sinhala-capable Llama-3 on Sri Lanka places data (Kaggle version)

Runs on a free Kaggle GPU (P100 or T4x2, 16GB+). Fine-tunes `ihalage/llama3-sinhala` - a Llama-3-8B-Instruct model already adapted for Sinhala - with QLoRA on the TripMe places instruction dataset, in **both English and Sinhala**, so the resulting model can hold the same conversation in either language.

**Why this base model:** the source places dataset itself has no Sinhala text (only English `description` fields), so a generic base model (Phi-3, vanilla Llama-3, Gemma) would need to learn Sinhala from scratch during this fine-tune - unrealistic with ~4k examples per language. `ihalage/llama3-sinhala` already understands and generates Sinhala; this fine-tune's job is narrower: teach it the TripMe voice and ground it in the Sri Lanka places data, in both languages.

**The full dataset (~37.6k examples) is too large for one Kaggle session, so training is split into 10 chunks, one per session.** See "Chunked training across sessions" below for the resume workflow.

**Before running (first session, chunk 1):**
1. Create a Kaggle Dataset named `tripme-data` containing `train.jsonl`, `val.jsonl`, `train_si.jsonl`, `val_si.jsonl` (from `data/training/` in the project).
2. Attach that dataset to this notebook (right sidebar > Add Input > search `tripme-data`).
3. In the right sidebar under **Settings**, set **Accelerator** to a GPU (P100, or T4 x2) and **Persistence** to "Files only" so checkpoints survive if the session restarts.
4. Run all cells, then click **Save Version** to persist the output (adapter + `chunk_state.json`).

**Before running (session 2 onward):**
1. Download the previous session's output, and upload it as a new Kaggle Dataset (e.g. `tripme-checkpoint`) - or a new version of an existing one.
2. Attach **both** `tripme-data` and `tripme-checkpoint` as inputs to a fresh copy of this notebook.
3. Run all cells - it will detect the checkpoint, resume the LoRA weights, and train the next chunk automatically.

Unlike Colab, there's no Drive mount step - Kaggle mounts your attached dataset automatically at `/kaggle/input/tripme-data/`, and this notebook's own output directory `/kaggle/working/` persists across a session and downloads as a zip via "Save Version" when you're done.

In [ ]:
import os
import json
import math

# Kaggle nests attached datasets under a path that includes your username
# (e.g. /kaggle/input/datasets/<username>/tripme-data), not always the flat
# /kaggle/input/tripme-data/ shape - search for wherever train.jsonl actually
# landed rather than hardcoding one specific path shape.
KAGGLE_INPUT_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.jsonl" in files:
        KAGGLE_INPUT_DIR = root
        break

assert KAGGLE_INPUT_DIR is not None, (
    "Could not find train.jsonl anywhere under /kaggle/input - attach the "
    "'tripme-data' dataset via the right sidebar (Add Input) before running "
    "the rest of this notebook."
)
print("Using data directory:", KAGGLE_INPUT_DIR)
print("Found:", os.listdir(KAGGLE_INPUT_DIR))

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets
!pip install -q "trl==1.6.0"

In [ ]:
DATA_DIR = KAGGLE_INPUT_DIR
CHECKPOINT_DIR = "/kaggle/working/tripme-checkpoints"
OUTPUT_DIR = "/kaggle/working/tripme-adapter"
BASE_MODEL = "ihalage/llama3-sinhala"  # Llama-3-8B-Instruct fine-tuned for Sinhala (Apache 2.0)
MAX_SEQ_LENGTH = 320  # dataset responses run ~25-115 words (max ~154); 320 tokens covers system+user+assistant with safe headroom, faster than 512
# 15 not 10: actual throughput on a T4 turned out to be ~2.5 min/step
# (not the ~1.5 min/step originally assumed), so a 1/10th chunk (~361
# steps) was projected at ~15h - well past the 9h safety stop, meaning a
# single chunk could never finish in one session. Smaller chunks (~240
# steps, ~10h projected... still tight, but each individual save point
# comes sooner, so a lost/cancelled Quick Save costs less progress) trade
# more total sessions for a much higher chance any given session actually
# reaches a chunk boundary and a clean save.
NUM_CHUNKS = 15

# Look for a previous run's output (chunk_state.json + saved adapter) attached
# as an input dataset (e.g. "tripme-checkpoint"), separate from the main
# "tripme-data" dataset which only has the jsonl files. If found, that's where
# we resume chunk progress and LoRA weights from.
PREV_RUN_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "chunk_state.json" in files:
        PREV_RUN_DIR = root
        break
if PREV_RUN_DIR:
    print("Found previous run output at:", PREV_RUN_DIR)
else:
    print("No previous run found - this will start from chunk 1.")

In [ ]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# Llama-3 attention/MLP module names (different from Phi-3's qkv_proj/gate_up_proj naming)
if PREV_RUN_DIR:
    prev_adapter_dir = os.path.join(PREV_RUN_DIR, "tripme-adapter")
    assert os.path.isdir(prev_adapter_dir), (
        f"chunk_state.json found at {PREV_RUN_DIR} but no tripme-adapter/ "
        "folder next to it - make sure the checkpoint dataset includes both."
    )
    print("Resuming LoRA weights from:", prev_adapter_dir)
    model = PeftModel.from_pretrained(model, prev_adapter_dir, is_trainable=True)
else:
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",   # attention
            "gate_proj", "up_proj", "down_proj",       # MLP
        ],
    )
    model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Load and combine English + Sinhala datasets

Both language datasets are trained together in one pass, so the model learns to answer in whichever language it's addressed in, using the same TripMe voice and the same underlying place facts.

## Chunked training across sessions

Training the full ~37.6k combined dataset in one Kaggle session would take well over the 9-12 hour session limit. Instead this notebook splits the shuffled training set into `NUM_CHUNKS` equal pieces and trains **one chunk per session**:

1. Run the notebook. It trains on chunk 1 and saves the LoRA adapter + a `chunk_state.json` marker to `/kaggle/working/`.
2. Click **Save Version** to persist the output.
3. Create a **new Kaggle Dataset** from that output (or a new version of one), named e.g. `tripme-checkpoint`, and attach it as an *additional* input to the notebook alongside `tripme-data`.
4. Re-run the notebook. It finds `chunk_state.json` and the previous adapter under `/kaggle/input/`, resumes the LoRA weights, and trains on chunk 2.
5. Repeat until `next_chunk` reaches `NUM_CHUNKS` (10 sessions total).

Each chunk trains on genuinely new examples (thanks to the fixed `seed=42` shuffle), and the adapter carries forward everything learned in earlier chunks, so this is equivalent to one long training run split across sessions - not 10 independent partial trainings.

In [ ]:
dataset_en = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/val.jsonl",
    },
)
dataset_si = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train_si.jsonl",
        "validation": f"{DATA_DIR}/val_si.jsonl",
    },
)

train_combined = concatenate_datasets([dataset_en["train"], dataset_si["train"]]).shuffle(seed=42)
val_combined = concatenate_datasets([dataset_en["validation"], dataset_si["validation"]]).shuffle(seed=42)

# Chunked training: the fixed seed=42 shuffle means each chunk index always
# maps to the same slice of examples, so resuming picks up exactly where the
# previous session left off, with no overlap and no gaps.
total_train = len(train_combined)
chunk_size = math.ceil(total_train / NUM_CHUNKS)

if PREV_RUN_DIR:
    with open(os.path.join(PREV_RUN_DIR, "chunk_state.json")) as f:
        chunk_state = json.load(f)
else:
    chunk_state = {"next_chunk": 0}

current_chunk = chunk_state["next_chunk"]
assert current_chunk < NUM_CHUNKS, (
    f"All {NUM_CHUNKS} chunks already trained (next_chunk={current_chunk}). "
    "This run is done - nothing left to train."
)

chunk_start = current_chunk * chunk_size
chunk_end = min(chunk_start + chunk_size, total_train)
train_combined = train_combined.select(range(chunk_start, chunk_end))

print(f"Training chunk {current_chunk + 1}/{NUM_CHUNKS}  (examples {chunk_start}-{chunk_end} of {total_train})")
print(f"val:   {len(val_combined)} examples ({len(dataset_en['validation'])} en + {len(dataset_si['validation'])} si)")
print(train_combined[0])

In [ ]:
def format_example(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_combined = train_combined.map(format_example, remove_columns=train_combined.column_names)
val_combined = val_combined.map(format_example, remove_columns=val_combined.column_names)
print(train_combined[0]["text"][:500])

In [ ]:
import time
from transformers import TrainerCallback

MAX_RUNTIME_HOURS = 9.0  # safety stop well inside Kaggle's 9-12h session limit

class TimeLimitCallback(TrainerCallback):
    """Stops training cleanly at a step boundary once MAX_RUNTIME_HOURS has
    elapsed, so the session always has time left to save the adapter and
    chunk_state.json before Kaggle cuts it off - losing partial progress on
    this chunk is much better than losing the whole session's output."""

    def __init__(self, max_hours):
        self.deadline = time.time() + max_hours * 3600

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() >= self.deadline:
            print(f"\nHit {MAX_RUNTIME_HOURS}h safety limit at step {state.global_step} - stopping training cleanly.")
            control.should_training_stop = True
        return control

# warmup_ratio deliberately omitted - the installed trl version's SFTConfig
# (which Kaggle resolves at install time, not necessarily the pinned
# trl==1.6.0 if a base-image copy was already imported first) has dropped
# it in some releases, raising a TypeError. It's a minor LR-schedule nicety,
# not something worth pinning an exact trl build over - safe to leave out.
#
# per_device_train_batch_size=16, gradient_accumulation_steps=1 (was 8/2):
# same effective batch size (16) so training dynamics are unchanged, but
# halves the step count per chunk - a session-1 run showed the T4 was only
# using ~12.6/15GB, so there was headroom to fold the accumulation loop
# into a single larger forward/backward pass and cut per-step overhead
# (logging/scheduler/accumulation bookkeeping that happens once per step).
sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=150,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_combined,
    eval_dataset=val_combined,
    processing_class=tokenizer,
    callbacks=[TimeLimitCallback(MAX_RUNTIME_HOURS)],
)

## Train

Each session trains on one chunk only (~3.8k combined examples), which should take well under the 9-12 hour Kaggle session limit. If a single session is itself interrupted mid-chunk, just re-run all cells above and this one - it auto-resumes from the latest step checkpoint under `/kaggle/working/tripme-checkpoints` (persistence must be set to "Files only" in the notebook's Settings sidebar - see the note at the top). This is separate from the cross-session chunk resume described above: this checkpoint only covers finishing the *current* chunk, not advancing to the next one.

In [ ]:
resume = any(d.startswith("checkpoint-") for d in os.listdir(CHECKPOINT_DIR)) \
    if os.path.isdir(CHECKPOINT_DIR) else False

train_result = trainer.train(resume_from_checkpoint=resume)

chunk_finished = trainer.state.global_step >= trainer.state.max_steps
if not chunk_finished:
    print(f"\nStopped early by the time limit at step {trainer.state.global_step}/{trainer.state.max_steps} "
          f"- chunk {current_chunk + 1} is NOT fully trained yet. Re-run this same chunk next session "
          "(attaching this session's output as tripme-checkpoint) to finish it before moving on.")
else:
    print(f"\nChunk {current_chunk + 1}/{NUM_CHUNKS} finished all {trainer.state.max_steps} steps.")

In [ ]:
# Save the LoRA adapter and advance the chunk marker to /kaggle/working/ -
# click "Save Version" (top right) after this finishes to persist it as a
# downloadable Kaggle Output. Upload that output as a new Kaggle Dataset (or
# a new version of one) and attach it as an input on the next session to
# resume training.
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Only advance to the next chunk if this chunk actually finished all its
# steps. If the time-limit safety stop cut training short, next_chunk stays
# the same, so next session's step-checkpoint resume (see CHECKPOINT_DIR
# above) picks up mid-chunk instead of skipping the rest of these examples.
next_chunk_value = current_chunk + 1 if chunk_finished else current_chunk
with open("/kaggle/working/chunk_state.json", "w") as f:
    json.dump({"next_chunk": next_chunk_value}, f)

print(f"Adapter saved to {OUTPUT_DIR}")
if not chunk_finished:
    print(f"Chunk {current_chunk + 1}/{NUM_CHUNKS} only partially trained - next session will resume this same chunk.")
elif next_chunk_value >= NUM_CHUNKS:
    print("All chunks trained! This adapter has now seen the full dataset.")
else:
    print(f"Chunk {current_chunk + 1}/{NUM_CHUNKS} complete. {NUM_CHUNKS - next_chunk_value} chunk(s) remaining - "
          "attach this output as a new input dataset next session to continue.")

## Manual evaluation

Sanity-check the fine-tuned model on realistic queries in both languages before trusting it. Look for: (1) groundedness - does it mention real places/facts from the dataset, not invented ones; (2) tone - does it sound like a warm local guide, not a generic chatbot; (3) does the Sinhala read as natural, grammatically correct formal Sinhala (not stitched/awkward); (4) it should decline or hedge gracefully on questions outside the dataset's scope.

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer, max_new_tokens=250)

SYSTEM_PROMPT_EN = (
    "You are TripMe, a warm and knowledgeable Sri Lankan travel voice assistant. "
    "Reply naturally in English, using only the facts provided about each place."
)
SYSTEM_PROMPT_SI = (
'ඔබ TripMe නම් වූ, ශ්\u200dරී ලංකාවේ සංචාරක ස්ථාන පිළිබඳ නිර්දේශ ලබා දෙන කථන සහායකයෙකි. ලබා දී ඇති කරුණු පමණක් භාවිතා කරමින්, පිරිසිදු හා විධිමත් සිංහල භාෂාවෙන් ස්වාභාවිකව හා උණුසුම්ව පිළිතුරු දෙන්න.'
)

eval_prompts = [
    (SYSTEM_PROMPT_EN, "What's worth visiting near Kandy?"),
    (SYSTEM_PROMPT_EN, "Is Dunhinda safe to visit right now?"),
    (SYSTEM_PROMPT_EN, "Can you plan a day trip around Nuwara Eliya for me?"),
    (SYSTEM_PROMPT_EN, "Play the audio guide for Dunhinda."),
    (SYSTEM_PROMPT_EN,
     "My trip: 3 days in Kandy, day 1 just finished (2 days left). Total budget: 35000 LKR. "
     "Spent so far: 8000 LKR (transport 1300, food 3900, tickets 1400, stay 1400).\n\n"
     "Am I on track with my spending?"),
    (SYSTEM_PROMPT_SI, "Kandy ආසන්නයේ නැරඹීමට සුදුසු ස්ථානයක් තිබේද?"),
    (SYSTEM_PROMPT_SI, "Dunhinda දැන් යාමට ආරක්ෂිතද?"),
    (SYSTEM_PROMPT_SI, "Nuwara Eliya අවට එක්දින සංචාරයක් සැලසුම් කර දෙන්නද?"),
    (SYSTEM_PROMPT_SI, "Dunhinda ගැන කථන මාර්ගෝපදේශය අරඹන්න."),
]

for system_prompt, prompt in eval_prompts:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = gen(formatted, do_sample=True, temperature=0.7, top_p=0.9)[0]["generated_text"]
    response = output[len(formatted):].strip()
    print(f"Q: {prompt}\nA: {response}\n{'-'*80}")